In [ ]:
# Cell 1: 실행 환경 및 라이브러리 준비

# --- 라이브러리 import ---
# pandas: IAM / 사용자 메타데이터 / 행동 로그를 DataFrame 형태로 처리
# numpy: AI 입력용 수치 데이터와 계산 처리
# random: 재현 가능한 Mock Data 생성 (seed 고정)
# networkx: IAM 권한 관계 및 권한 상승 경로 그래프 분석
# matplotlib: 기본 시각화 지원
# plotly: 공모전 시연용 인터랙티브 그래프 및 Dashboard 시각화
# ipywidgets: Colab 환경에서 버튼 및 인터랙션 구현
# IsolationForest: 사용자 행동 이상 탐지 AI 모델

import sys
import random

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

import sklearn
from sklearn.ensemble import IsolationForest

# --- 재현 가능한 실험 환경 설정 (Seed 고정) ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"Random seeds set to {SEED} for reproducibility.")

# --- 주요 라이브러리 버전 확인 ---
print("\n--- Installed Library Versions ---")
print(f"Python version: {sys.version.split(' ')[0]}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"networkx version: {nx.__version__}")

# --- 환경 설정 완료 메시지 ---
print("\nCell 1 environment setup completed successfully.")

In [ ]:

print("--- Cell 12-1B: Execute Remediation Application Simulation ---")

remediation_application_state = run_remediation_application_simulation(
    actions_df=remediation_actions_df,
    policy_data=mock_iam_policy_data,
    metadata_df=mock_identity_security_metadata,
    extract_statements_func=extract_statements,
)

# Cell 12-2 이후와 호환되는 기존 변수명 유지
cell12_original_iam_snapshot = remediation_application_state[
    "original_policy_snapshot"
]
cell12_original_metadata_snapshot = remediation_application_state[
    "original_metadata_snapshot"
]
remediated_iam_policy_data = remediation_application_state[
    "remediated_policy_data"
]
remediated_identity_metadata = remediation_application_state[
    "remediated_metadata_df"
]
simulation_rows = remediation_application_state["simulation_rows"]
policy_application_logs = remediation_application_state["policy_logs"]
metadata_application_logs = remediation_application_state["metadata_logs"]
policy_application_log_df = remediation_application_state["policy_log_df"]
metadata_application_log_df = remediation_application_state["metadata_log_df"]
remediation_application_summary_df = remediation_application_state[
    "summary_df"
]

policy_log_columns = POLICY_LOG_COLUMNS.copy()
metadata_log_columns = METADATA_LOG_COLUMNS.copy()

summary_row = remediation_application_summary_df.iloc[0]
policy_change_targets = int(summary_row["policy_change_targets"])
metadata_change_targets = int(summary_row["metadata_change_targets"])
total_removed_actions = int(summary_row["total_removed_actions"])

print("\n1. Policy Application Log")
display(policy_application_log_df)

print("\n2. Metadata Application Log")
display(metadata_application_log_df)

print("\n3. Application Summary")
display(remediation_application_summary_df)

print("Original Mock Data remains unchanged.")
print("All Cell 12-1 output contracts verified.")
print(
    "Cell 12-1 Generic Remediation Application "
    "Simulation completed successfully."
)


In [ ]:
import pandas as pd

print("--- Cell 12-2A: Post-remediation Rule Analysis Functions ---")


REQUIRED_POST_RULE_STATE_KEYS = [
    "rule_engine",
    "input_df",
    "analysis_df",
    "findings_df",
    "identity_summary_df",
    "trigger_counts",
    "metrics",
]


def validate_post_rule_inputs(
    policy_data,
    identity_metadata_df,
    normalize_policy_dataset_func,
    run_rule_analysis_func,
):
    """Cell 12-2의 입력 데이터와 재사용 함수를 검증합니다."""
    if not isinstance(policy_data, list):
        raise TypeError("policy_data는 list여야 합니다.")
    if not isinstance(identity_metadata_df, pd.DataFrame):
        raise TypeError("identity_metadata_df는 DataFrame이어야 합니다.")
    if identity_metadata_df.empty:
        raise ValueError("identity_metadata_df가 비어 있습니다.")
    if not callable(normalize_policy_dataset_func):
        raise TypeError("normalize_policy_dataset_func는 함수여야 합니다.")
    if not callable(run_rule_analysis_func):
        raise TypeError("run_rule_analysis_func는 함수여야 합니다.")

    required_metadata_columns = [
        "identity",
        "cloud_provider",
        "is_admin",
        "mfa_enabled",
        "has_api_key",
    ]
    missing_columns = [
        column
        for column in required_metadata_columns
        if column not in identity_metadata_df.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Identity metadata 필수 컬럼 누락: {missing_columns}"
        )
    if identity_metadata_df["identity"].duplicated().any():
        raise ValueError("Identity metadata에 중복 identity가 있습니다.")


def merge_triggered_rules(rule_lists):
    """여러 레코드에서 발동한 Rule 이름을 중복 없이 정렬합니다."""
    return sorted({
        rule_name
        for rule_list in rule_lists
        for rule_name in rule_list
    })


def complete_rule_identity_summary(
    grouped_rule_summary_df,
    identity_metadata_df,
):
    """정책 레코드가 사라진 Identity까지 Rule 요약에 포함합니다."""
    identity_base_df = (
        identity_metadata_df[[
            "identity",
            "cloud_provider",
        ]]
        .rename(columns={"cloud_provider": "source_csp"})
    )

    identity_summary_df = identity_base_df.merge(
        grouped_rule_summary_df,
        on=["identity", "source_csp"],
        how="left",
        validate="one_to_one",
    )

    identity_summary_df["max_rule_score"] = (
        identity_summary_df["max_rule_score"].fillna(0.0)
    )
    identity_summary_df["violation_record_count"] = (
        identity_summary_df["violation_record_count"]
        .fillna(0)
        .astype(int)
    )
    identity_summary_df["triggered_rule_names"] = (
        identity_summary_df["triggered_rule_names"].apply(
            lambda value: value if isinstance(value, list) else []
        )
    )

    return identity_base_df, identity_summary_df


def create_post_rule_summary(normalized_df, rule_findings_df):
    """조치 후 정규화 및 Rule 분석 핵심 지표를 한 행으로 만듭니다."""
    return pd.DataFrame([{
        "normalized_records": len(normalized_df),
        "wildcard_records": int(normalized_df["is_wildcard"].sum()),
        "unknown_records": int(
            (normalized_df["operation"] == "UNKNOWN").sum()
        ),
        "rule_violation_records": len(rule_findings_df),
        "identities_with_violations": (
            rule_findings_df["identity"].nunique()
        ),
    }])


def validate_post_rule_outputs(
    normalized_records,
    normalized_df,
    rule_state,
    identity_summary_df,
    summary_df,
):
    """Cell 12-3 이후가 사용할 출력 계약을 검증합니다."""
    if not isinstance(normalized_records, list):
        raise TypeError("normalized_records는 list여야 합니다.")
    if not isinstance(normalized_df, pd.DataFrame):
        raise TypeError("normalized_df는 DataFrame이어야 합니다.")

    missing_state_keys = [
        key
        for key in REQUIRED_POST_RULE_STATE_KEYS
        if key not in rule_state
    ]
    if missing_state_keys:
        raise ValueError(
            f"Rule 분석 결과 key 누락: {missing_state_keys}"
        )

    dataframe_keys = [
        "input_df",
        "analysis_df",
        "findings_df",
        "identity_summary_df",
    ]
    for key in dataframe_keys:
        if not isinstance(rule_state[key], pd.DataFrame):
            raise TypeError(f"rule_state['{key}']는 DataFrame이어야 합니다.")

    if not isinstance(identity_summary_df, pd.DataFrame):
        raise TypeError("identity_summary_df는 DataFrame이어야 합니다.")
    if not isinstance(summary_df, pd.DataFrame):
        raise TypeError("summary_df는 DataFrame이어야 합니다.")
    if identity_summary_df["identity"].duplicated().any():
        raise ValueError("identity_summary_df는 identity당 한 행이어야 합니다.")


def run_post_remediation_rule_analysis(
    policy_data,
    identity_metadata_df,
    normalize_policy_dataset_func,
    run_rule_analysis_func,
):
    """조치 후 정책을 정규화하고 기존 Rule 분석 모듈로 재분석합니다."""
    validate_post_rule_inputs(
        policy_data,
        identity_metadata_df,
        normalize_policy_dataset_func,
        run_rule_analysis_func,
    )

    normalized_records = normalize_policy_dataset_func(policy_data)
    normalized_df = pd.DataFrame(normalized_records)
    if normalized_df.empty:
        raise ValueError("Remediation 이후 정규화 결과가 비어 있습니다.")

    rule_state = run_rule_analysis_func(
        normalized_df=normalized_df,
        identity_metadata_df=identity_metadata_df,
    )

    grouped_rule_summary_df = rule_state["identity_summary_df"]
    identity_base_df, identity_summary_df = (
        complete_rule_identity_summary(
            grouped_rule_summary_df,
            identity_metadata_df,
        )
    )
    summary_df = create_post_rule_summary(
        normalized_df,
        rule_state["findings_df"],
    )

    validate_post_rule_outputs(
        normalized_records,
        normalized_df,
        rule_state,
        identity_summary_df,
        summary_df,
    )

    return {
        "normalized_records": normalized_records,
        "normalized_df": normalized_df,
        "rule_engine": rule_state["rule_engine"],
        "rule_input_df": rule_state["input_df"],
        "rule_analysis_df": rule_state["analysis_df"],
        "rule_findings_df": rule_state["findings_df"],
        "rule_grouped_df": grouped_rule_summary_df,
        "identity_base_df": identity_base_df,
        "rule_identity_summary_df": identity_summary_df,
        "rule_trigger_counts": rule_state["trigger_counts"],
        "rule_metrics": rule_state["metrics"],
        "rule_summary_df": summary_df,
    }


print("Cell 12-2A completed: post-remediation Rule functions defined.")


In [ ]:

print("--- Cell 12-2B: Execute Post-remediation Rule Analysis ---")

post_rule_state = run_post_remediation_rule_analysis(
    policy_data=remediated_iam_policy_data,
    identity_metadata_df=remediated_identity_metadata,
    normalize_policy_dataset_func=normalize_policy_dataset,
    run_rule_analysis_func=run_rule_analysis,
)

# Cell 12-3 이후와 호환되는 기존 변수명 유지
post_normalized_records = post_rule_state["normalized_records"]
post_normalized_df = post_rule_state["normalized_df"]
post_rule_engine = post_rule_state["rule_engine"]
post_rule_input_df = post_rule_state["rule_input_df"]
post_rule_analysis_df = post_rule_state["rule_analysis_df"]
post_rule_findings_df = post_rule_state["rule_findings_df"]
post_rule_grouped_df = post_rule_state["rule_grouped_df"]
post_identity_base_df = post_rule_state["identity_base_df"]
post_rule_identity_summary_df = post_rule_state[
    "rule_identity_summary_df"
]
post_rule_trigger_counts = post_rule_state["rule_trigger_counts"]
post_rule_metrics = post_rule_state["rule_metrics"]
post_rule_summary_df = post_rule_state["rule_summary_df"]

print("\n1. Post-remediation Normalization")
print("---------------------------------")
print(f"Normalized Records: {len(post_normalized_df)}")
print(
    f"Wildcard Records: "
    f"{int(post_normalized_df['is_wildcard'].sum())}"
)
print(
    f"UNKNOWN Records: "
    f"{int((post_normalized_df['operation'] == 'UNKNOWN').sum())}"
)

print("\n2. Post-remediation Rule Analysis")
print("---------------------------------")
print(f"Rule Violation Records: {len(post_rule_findings_df)}")
print(
    "Identities with Rule Violations: "
    f"{post_rule_findings_df['identity'].nunique()}"
)

print("\n3. Post-remediation Rule Summary")
print("--------------------------------")
display(post_rule_summary_df)

if post_rule_findings_df.empty:
    print("No Rule violations remain after remediation.")
else:
    display(
        post_rule_findings_df[[
            "identity",
            "source_csp",
            "rule_score",
            "triggered_rules",
        ]]
    )

print("All Cell 12-2 output contracts verified.")
print(
    "Cell 12-2 Post-remediation Normalization "
    "and Rule Re-analysis completed successfully."
)


In [ ]:
import pandas as pd
import networkx as nx

print("--- Cell 12-3A: Post-remediation Graph Analysis Functions ---")


POST_GRAPH_COUNT_COLUMNS = [
    "permission_edge_count",
    "wildcard_edge_count",
    "unknown_operation_count",
    "target_count",
]

POST_GRAPH_FINDING_COLUMNS = [
    "identity",
    "finding_type",
    "operation",
    "target",
    "effect",
    "source_csp",
    "description",
]

POST_RAW_RULE_COLUMNS = [
    "raw_action",
    "finding_type",
    "severity",
    "description",
]

POST_RAW_PRIVILEGE_COLUMNS = [
    "identity",
    "source_csp",
    "raw_action",
    "scope",
    "effect",
    "finding_type",
    "severity",
    "description",
]

REQUIRED_POST_GRAPH_STATE_KEYS = [
    "graph",
    "identity_summary_df",
]


def validate_post_graph_inputs(
    records,
    normalized_df,
    policy_data,
    identity_metadata_df,
    raw_privilege_findings_df,
    run_graph_analysis_func,
    extract_statements_func,
):
    """Cell 12-3 입력 데이터와 재사용 함수를 검증합니다."""
    if not isinstance(records, list):
        raise TypeError("records는 list여야 합니다.")
    if not isinstance(normalized_df, pd.DataFrame):
        raise TypeError("normalized_df는 DataFrame이어야 합니다.")
    if not isinstance(policy_data, list):
        raise TypeError("policy_data는 list여야 합니다.")
    if not isinstance(identity_metadata_df, pd.DataFrame):
        raise TypeError("identity_metadata_df는 DataFrame이어야 합니다.")
    if not isinstance(raw_privilege_findings_df, pd.DataFrame):
        raise TypeError("raw_privilege_findings_df는 DataFrame이어야 합니다.")
    if not callable(run_graph_analysis_func):
        raise TypeError("run_graph_analysis_func는 함수여야 합니다.")
    if not callable(extract_statements_func):
        raise TypeError("extract_statements_func는 함수여야 합니다.")

    missing_raw_rule_columns = [
        column
        for column in POST_RAW_RULE_COLUMNS
        if column not in raw_privilege_findings_df.columns
    ]
    if missing_raw_rule_columns:
        raise ValueError(
            "Raw privilege rule 필수 컬럼 누락: "
            f"{missing_raw_rule_columns}"
        )


def keep_post_graph_node_contract(graph):
    """조치 후 그래프 노드에는 기존 계약인 node_type만 유지합니다."""
    for node_name in graph.nodes:
        node_type = graph.nodes[node_name].get("node_type", "Resource")
        graph.nodes[node_name].clear()
        graph.nodes[node_name]["node_type"] = node_type
    return graph


def build_post_graph_identity_summary(
    graph_identity_summary_df,
    identity_metadata_df,
):
    """그래프 통계를 전체 Identity 기준표와 결합합니다."""
    grouped_df = (
        graph_identity_summary_df[[
            "identity",
            *POST_GRAPH_COUNT_COLUMNS,
        ]]
        .sort_values("identity")
        .reset_index(drop=True)
    )

    identity_base_df = (
        identity_metadata_df[[
            "identity",
            "cloud_provider",
        ]]
        .rename(columns={"cloud_provider": "source_csp"})
    )

    identity_summary_df = identity_base_df.merge(
        grouped_df,
        on="identity",
        how="left",
        validate="one_to_one",
    )
    identity_summary_df[POST_GRAPH_COUNT_COLUMNS] = (
        identity_summary_df[POST_GRAPH_COUNT_COLUMNS]
        .fillna(0)
        .astype(int)
    )

    return grouped_df, identity_base_df, identity_summary_df


def build_post_graph_findings(normalized_df):
    """조치 후 남은 Wildcard와 UNKNOWN operation을 Finding으로 만듭니다."""
    finding_frames = []

    wildcard_findings_df = normalized_df[
        normalized_df["is_wildcard"] == True
    ].copy()
    if not wildcard_findings_df.empty:
        wildcard_findings_df["finding_type"] = "WILDCARD_PERMISSION"
        wildcard_findings_df["description"] = (
            "Wildcard permission remains after remediation."
        )
        finding_frames.append(wildcard_findings_df)

    unknown_findings_df = normalized_df[
        normalized_df["operation"] == "UNKNOWN"
    ].copy()
    if not unknown_findings_df.empty:
        unknown_findings_df["finding_type"] = "UNKNOWN_OPERATION"
        unknown_findings_df["description"] = (
            "An unmapped operation remains after remediation."
        )
        finding_frames.append(unknown_findings_df)

    if finding_frames:
        findings_df = pd.concat(
            finding_frames,
            ignore_index=True,
        )[POST_GRAPH_FINDING_COLUMNS]
    else:
        findings_df = pd.DataFrame(columns=POST_GRAPH_FINDING_COLUMNS)

    return (
        wildcard_findings_df,
        unknown_findings_df,
        finding_frames,
        findings_df,
    )


def build_raw_privilege_rule_catalog(raw_privilege_findings_df):
    """조치 전 Raw Finding에서 재분석용 Rule catalog를 만듭니다."""
    rule_catalog_df = (
        raw_privilege_findings_df[POST_RAW_RULE_COLUMNS]
        .drop_duplicates(
            subset=["raw_action", "finding_type"]
        )
    )

    rules_by_action = {}
    for _, rule_row in rule_catalog_df.iterrows():
        action_key = str(rule_row["raw_action"]).lower()
        rules_by_action.setdefault(action_key, []).append(
            rule_row.to_dict()
        )

    return rule_catalog_df, rules_by_action


def detect_post_raw_privilege_findings(
    policy_data,
    rules_by_action,
    extract_statements_func,
):
    """조치 후 Raw policy에 Rule catalog의 위험 Action이 남았는지 확인합니다."""
    findings = []

    for policy_item in policy_data:
        statements = extract_statements_func(
            policy_item["csp"],
            policy_item["raw_policy_document"],
        )

        for statement in statements:
            for action in statement["actions"]:
                matched_rules = rules_by_action.get(
                    str(action).lower(),
                    [],
                )

                for matched_rule in matched_rules:
                    findings.append({
                        "identity": policy_item["identity"],
                        "source_csp": policy_item["csp"],
                        "raw_action": str(action),
                        "scope": statement["scope"],
                        "effect": statement["effect"],
                        "finding_type": matched_rule["finding_type"],
                        "severity": matched_rule["severity"],
                        "description": matched_rule["description"],
                    })

    findings_df = pd.DataFrame(
        findings,
        columns=POST_RAW_PRIVILEGE_COLUMNS,
    )
    return findings, findings_df


def create_post_graph_summary(
    graph,
    identity_summary_df,
    graph_findings_df,
    raw_privilege_findings_df,
):
    """조치 후 그래프 재분석 핵심 지표를 한 행으로 만듭니다."""
    return pd.DataFrame([{
        "graph_nodes": graph.number_of_nodes(),
        "graph_edges": graph.number_of_edges(),
        "wildcard_edges": int(
            identity_summary_df["wildcard_edge_count"].sum()
        ),
        "unknown_edges": int(
            identity_summary_df["unknown_operation_count"].sum()
        ),
        "graph_findings": len(graph_findings_df),
        "raw_privilege_findings": len(raw_privilege_findings_df),
    }])


def validate_post_graph_outputs(
    graph,
    identity_summary_df,
    graph_findings_df,
    raw_privilege_findings_df,
    confirmed_privilege_paths,
    summary_df,
):
    """Cell 12-4가 사용할 그래프 출력 계약을 검증합니다."""
    if not isinstance(graph, nx.MultiDiGraph):
        raise TypeError("graph는 nx.MultiDiGraph여야 합니다.")

    dataframe_outputs = {
        "identity_summary_df": identity_summary_df,
        "graph_findings_df": graph_findings_df,
        "raw_privilege_findings_df": raw_privilege_findings_df,
        "summary_df": summary_df,
    }
    for output_name, output_df in dataframe_outputs.items():
        if not isinstance(output_df, pd.DataFrame):
            raise TypeError(f"{output_name}는 DataFrame이어야 합니다.")

    if not isinstance(confirmed_privilege_paths, list):
        raise TypeError("confirmed_privilege_paths는 list여야 합니다.")
    if identity_summary_df["identity"].duplicated().any():
        raise ValueError("identity_summary_df는 identity당 한 행이어야 합니다.")


def run_post_remediation_graph_analysis(
    records,
    normalized_df,
    policy_data,
    identity_metadata_df,
    raw_privilege_findings_df,
    run_graph_analysis_func,
    extract_statements_func,
):
    """기존 Graph 모듈을 재사용해 조치 후 상태를 다시 분석합니다."""
    validate_post_graph_inputs(
        records,
        normalized_df,
        policy_data,
        identity_metadata_df,
        raw_privilege_findings_df,
        run_graph_analysis_func,
        extract_statements_func,
    )

    graph_state = run_graph_analysis_func(
        records=records,
        normalized_df=normalized_df,
        policy_data=policy_data,
        metadata_df=identity_metadata_df,
    )
    missing_graph_state_keys = [
        key
        for key in REQUIRED_POST_GRAPH_STATE_KEYS
        if key not in graph_state
    ]
    if missing_graph_state_keys:
        raise ValueError(
            f"Graph 분석 결과 key 누락: {missing_graph_state_keys}"
        )

    graph = keep_post_graph_node_contract(graph_state["graph"])
    grouped_df, identity_base_df, identity_summary_df = (
        build_post_graph_identity_summary(
            graph_state["identity_summary_df"],
            identity_metadata_df,
        )
    )

    (
        wildcard_findings_df,
        unknown_findings_df,
        finding_frames,
        graph_findings_df,
    ) = build_post_graph_findings(normalized_df)

    raw_rule_catalog_df, raw_rules_by_action = (
        build_raw_privilege_rule_catalog(raw_privilege_findings_df)
    )
    raw_findings, post_raw_findings_df = (
        detect_post_raw_privilege_findings(
            policy_data,
            raw_rules_by_action,
            extract_statements_func,
        )
    )

    # 현재 그래프는 Identity -> Target의 1-hop 구조입니다.
    confirmed_privilege_paths = []
    summary_df = create_post_graph_summary(
        graph,
        identity_summary_df,
        graph_findings_df,
        post_raw_findings_df,
    )

    validate_post_graph_outputs(
        graph,
        identity_summary_df,
        graph_findings_df,
        post_raw_findings_df,
        confirmed_privilege_paths,
        summary_df,
    )

    return {
        "graph": graph,
        "graph_state": graph_state,
        "graph_grouped_df": grouped_df,
        "identity_graph_base_df": identity_base_df,
        "graph_identity_summary_df": identity_summary_df,
        "wildcard_findings_df": wildcard_findings_df,
        "unknown_findings_df": unknown_findings_df,
        "graph_finding_frames": finding_frames,
        "graph_findings_df": graph_findings_df,
        "raw_rule_catalog_df": raw_rule_catalog_df,
        "raw_rules_by_action": raw_rules_by_action,
        "raw_privilege_findings": raw_findings,
        "raw_privilege_findings_df": post_raw_findings_df,
        "confirmed_privilege_paths": confirmed_privilege_paths,
        "graph_summary_df": summary_df,
    }


print("Cell 12-3A completed: post-remediation Graph functions defined.")



In [ ]:

print("--- Cell 12-3B: Execute Post-remediation Graph Analysis ---")

post_graph_state = run_post_remediation_graph_analysis(
    records=post_normalized_records,
    normalized_df=post_normalized_df,
    policy_data=remediated_iam_policy_data,
    identity_metadata_df=remediated_identity_metadata,
    raw_privilege_findings_df=raw_privilege_findings_df,
    run_graph_analysis_func=run_graph_analysis,
    extract_statements_func=extract_statements,
)

# Cell 12-4 이후와 호환되는 기존 변수명 유지
post_privilege_graph = post_graph_state["graph"]
post_graph_grouped_df = post_graph_state["graph_grouped_df"]
post_identity_graph_base_df = post_graph_state["identity_graph_base_df"]
post_graph_identity_summary_df = post_graph_state[
    "graph_identity_summary_df"
]
post_wildcard_findings_df = post_graph_state["wildcard_findings_df"]
post_unknown_findings_df = post_graph_state["unknown_findings_df"]
graph_finding_frames = post_graph_state["graph_finding_frames"]
post_graph_findings_df = post_graph_state["graph_findings_df"]
raw_rule_catalog_df = post_graph_state["raw_rule_catalog_df"]
raw_rules_by_action = post_graph_state["raw_rules_by_action"]
post_raw_privilege_findings = post_graph_state[
    "raw_privilege_findings"
]
post_raw_privilege_findings_df = post_graph_state[
    "raw_privilege_findings_df"
]
post_confirmed_privilege_paths = post_graph_state[
    "confirmed_privilege_paths"
]
post_graph_summary_df = post_graph_state["graph_summary_df"]

# 기존 상수성 변수명도 유지
identity_names = set(remediated_identity_metadata["identity"])
graph_count_columns = POST_GRAPH_COUNT_COLUMNS.copy()
graph_finding_columns = POST_GRAPH_FINDING_COLUMNS.copy()
raw_rule_columns = POST_RAW_RULE_COLUMNS.copy()
raw_privilege_columns = POST_RAW_PRIVILEGE_COLUMNS.copy()

print("\n1. Post-remediation Graph Construction")
print("--------------------------------------")
print(f"Graph Nodes: {post_privilege_graph.number_of_nodes()}")
print(f"Graph Edges: {post_privilege_graph.number_of_edges()}")

print("\n2. Graph Finding Re-analysis")
print("----------------------------")
print(f"Graph Findings: {len(post_graph_findings_df)}")
if not post_graph_findings_df.empty:
    display(post_graph_findings_df)

print("\n3. Raw Privilege Re-analysis")
print("----------------------------")
print(
    "Raw Privilege Findings: "
    f"{len(post_raw_privilege_findings_df)}"
)
if not post_raw_privilege_findings_df.empty:
    display(post_raw_privilege_findings_df)

print("\n4. Graph Re-analysis Summary")
print("----------------------------")
display(post_graph_summary_df)

print("All Cell 12-3 output contracts verified.")
print(
    "Cell 12-3 Post-remediation "
    "Graph Re-analysis completed successfully."
)


In [ ]:
import pandas as pd

print("--- Cell 12-4A: Post-remediation Risk Comparison Functions ---")


POST_RISK_REQUIRED_DATAFRAMES = {
    "integrated_risk_df",
    "risk_ranked_df",
    "ai_analysis_df",
    "post_rule_identity_summary_df",
    "post_graph_identity_summary_df",
    "post_graph_findings_df",
    "post_raw_privilege_findings_df",
    "remediated_identity_metadata",
    "remediation_actions_df",
}

POST_RISK_NUMERIC_COLUMNS = [
    "rule_score",
    "rule_finding_count",
    "permission_edge_count",
    "wildcard_edge_count",
    "unknown_operation_count",
    "target_count",
    "graph_finding_count",
    "raw_privilege_finding_count",
]

POST_RISK_INTEGER_COLUMNS = [
    "rule_finding_count",
    "permission_edge_count",
    "wildcard_edge_count",
    "unknown_operation_count",
    "target_count",
    "graph_finding_count",
    "raw_privilege_finding_count",
]

POST_RISK_LIST_COLUMNS = [
    "triggered_rules",
    "graph_finding_types",
    "raw_privilege_finding_types",
    "raw_privilege_actions",
]

POST_ACTION_ORDER = {
    "BLOCK": 0,
    "ALERT": 1,
    "ALLOW": 2,
}

RISK_COMPARISON_COLUMNS = [
    "identity",
    "source_csp",
    "before_rule_score",
    "after_rule_score",
    "before_graph_findings",
    "after_graph_findings",
    "before_risk_score",
    "after_risk_score",
    "risk_reduction",
    "before_action",
    "after_action",
    "risk_improved",
]


def validate_post_risk_inputs(
    dataframe_inputs,
    merge_module_class,
    run_integrated_risk_analysis_func,
):
    """Cell 12-4 입력 DataFrame과 재사용 모듈을 검증합니다."""
    missing_inputs = sorted(
        POST_RISK_REQUIRED_DATAFRAMES - set(dataframe_inputs)
    )
    if missing_inputs:
        raise ValueError(f"Cell 12-4 입력 누락: {missing_inputs}")

    invalid_inputs = [
        name
        for name, value in dataframe_inputs.items()
        if name in POST_RISK_REQUIRED_DATAFRAMES
        and not isinstance(value, pd.DataFrame)
    ]
    if invalid_inputs:
        raise TypeError(
            f"다음 입력은 DataFrame이어야 합니다: {invalid_inputs}"
        )
    if not callable(merge_module_class):
        raise TypeError("merge_module_class는 호출 가능해야 합니다.")
    if not callable(run_integrated_risk_analysis_func):
        raise TypeError(
            "run_integrated_risk_analysis_func는 함수여야 합니다."
        )


def prepare_post_graph_debug_outputs(
    graph_identity_summary_df,
    graph_findings_df,
    raw_privilege_findings_df,
):
    """기존 Cell 12-4 중간 Graph 집계 변수의 구조를 유지합니다."""
    graph_agg_df = graph_identity_summary_df[[
        "identity",
        "permission_edge_count",
        "wildcard_edge_count",
        "unknown_operation_count",
        "target_count",
    ]].copy()

    if graph_findings_df.empty:
        graph_finding_agg_df = pd.DataFrame(columns=[
            "identity",
            "graph_finding_count",
            "graph_finding_types",
        ])
    else:
        graph_finding_agg_df = (
            graph_findings_df.groupby("identity")
            .agg(
                graph_finding_count=("finding_type", "size"),
                graph_finding_types=(
                    "finding_type",
                    lambda values: sorted(set(values)),
                ),
            )
            .reset_index()
        )

    if raw_privilege_findings_df.empty:
        raw_finding_agg_df = pd.DataFrame(columns=[
            "identity",
            "raw_privilege_finding_count",
            "raw_privilege_finding_types",
            "raw_privilege_actions",
        ])
    else:
        raw_finding_agg_df = (
            raw_privilege_findings_df.groupby("identity")
            .agg(
                raw_privilege_finding_count=("finding_type", "size"),
                raw_privilege_finding_types=(
                    "finding_type",
                    lambda values: sorted(set(values)),
                ),
                raw_privilege_actions=(
                    "raw_action",
                    lambda values: sorted(set(values)),
                ),
            )
            .reset_index()
        )

    graph_agg_df = (
        graph_agg_df.merge(
            graph_finding_agg_df,
            on="identity",
            how="left",
            validate="one_to_one",
        )
        .merge(
            raw_finding_agg_df,
            on="identity",
            how="left",
            validate="one_to_one",
        )
    )

    return graph_agg_df, graph_finding_agg_df, raw_finding_agg_df


def build_post_decision_reasons(row):
    """조치 후 Hybrid 판정의 핵심 이유를 반환합니다."""
    reasons = []

    if row["final_action"] == "BLOCK" and row["decision_by"] == "rule":
        reasons.extend(row["triggered_rules"])
    elif row["final_action"] == "ALERT" and row["decision_by"] == "hybrid":
        reasons.extend(row["triggered_rules"])
        if bool(row["is_anomaly"]):
            reasons.append("AI_ANOMALY_DETECTED")

    return list(dict.fromkeys(reasons))


def build_post_raw_evidence(row):
    """조치 후 Raw policy Finding을 설명 가능한 문자열로 변환합니다."""
    evidence = []

    for finding_type in row["raw_privilege_finding_types"]:
        if row["raw_privilege_actions"]:
            evidence.extend([
                f"{finding_type}: {action}"
                for action in row["raw_privilege_actions"]
            ])
        else:
            evidence.append(finding_type)

    return list(dict.fromkeys(evidence))


def normalize_post_integrated_output(integrated_df):
    """재사용 모듈의 내부 정렬 key를 제거하고 기존 출력 계약을 맞춥니다."""
    result_df = integrated_df.drop(
        columns=["action_sort_key"],
        errors="ignore",
    ).copy()

    # 기존 Cell 12-4의 설명 생성 규칙을 명시적으로 유지합니다.
    result_df["decision_reasons"] = result_df.apply(
        build_post_decision_reasons,
        axis=1,
    )
    result_df["graph_evidence"] = result_df[
        "graph_finding_types"
    ].apply(list)
    result_df["raw_policy_evidence"] = result_df.apply(
        build_post_raw_evidence,
        axis=1,
    )
    return result_df


def rank_post_remediation_risk(integrated_df):
    """조치 후 Identity를 BLOCK→ALERT→ALLOW 및 위험도 순으로 정렬합니다."""
    return (
        integrated_df.assign(
            action_sort_key=integrated_df["final_action"].map(
                POST_ACTION_ORDER
            )
        )
        .sort_values(
            by=[
                "action_sort_key",
                "hybrid_risk_score",
                "is_anomaly",
                "graph_finding_count",
                "identity",
            ],
            ascending=[True, False, False, False, True],
        )
        .drop(columns=["action_sort_key"])
        .reset_index(drop=True)
    )


def build_risk_comparison(
    before_integrated_df,
    after_integrated_df,
    before_ranked_df,
    remediation_actions_df,
):
    """조치 전후 위험 점수·판정·우선순위를 Identity별로 비교합니다."""
    before_df = before_integrated_df[[
        "identity",
        "source_csp",
        "rule_score",
        "ai_anomaly_score_raw",
        "graph_finding_count",
        "hybrid_risk_score",
        "final_action",
    ]].rename(columns={
        "rule_score": "before_rule_score",
        "ai_anomaly_score_raw": "before_ai_score",
        "graph_finding_count": "before_graph_findings",
        "hybrid_risk_score": "before_risk_score",
        "final_action": "before_action",
    })

    after_df = after_integrated_df[[
        "identity",
        "source_csp",
        "rule_score",
        "ai_anomaly_score_raw",
        "graph_finding_count",
        "hybrid_risk_score",
        "final_action",
    ]].rename(columns={
        "rule_score": "after_rule_score",
        "ai_anomaly_score_raw": "after_ai_score",
        "graph_finding_count": "after_graph_findings",
        "hybrid_risk_score": "after_risk_score",
        "final_action": "after_action",
    })

    comparison_df = before_df.merge(
        after_df,
        on=["identity", "source_csp"],
        how="inner",
        validate="one_to_one",
    )

    applied_identities = set(
        remediation_actions_df.loc[
            remediation_actions_df["simulation_ready"]
            .fillna(False)
            .astype(bool),
            "identity",
        ]
    )
    comparison_df["remediation_applied"] = comparison_df[
        "identity"
    ].isin(applied_identities)
    comparison_df["risk_reduction"] = (
        comparison_df["before_risk_score"]
        - comparison_df["after_risk_score"]
    ).round(3)
    comparison_df["action_changed"] = (
        comparison_df["before_action"]
        != comparison_df["after_action"]
    )
    comparison_df["risk_improved"] = (
        comparison_df["after_risk_score"]
        < comparison_df["before_risk_score"]
    )

    before_rank_map = {
        identity: rank
        for rank, identity in enumerate(
            before_ranked_df["identity"],
            start=1,
        )
    }
    comparison_df["before_priority"] = comparison_df[
        "identity"
    ].map(before_rank_map)
    comparison_df.sort_values(by="before_priority", inplace=True)
    comparison_df.reset_index(drop=True, inplace=True)

    remediation_effect_df = comparison_df[
        comparison_df["remediation_applied"]
    ].copy()

    return {
        "before_df": before_df,
        "after_df": after_df,
        "applied_identities": applied_identities,
        "before_rank_map": before_rank_map,
        "comparison_df": comparison_df,
        "remediation_effect_df": remediation_effect_df,
    }


def create_before_after_summary(
    before_integrated_df,
    after_integrated_df,
    remediation_effect_df,
):
    """전체 조치 효과를 한 행의 요약 지표로 만듭니다."""
    return pd.DataFrame([{
        "before_block_count": int(
            (before_integrated_df["final_action"] == "BLOCK").sum()
        ),
        "after_block_count": int(
            (after_integrated_df["final_action"] == "BLOCK").sum()
        ),
        "before_average_risk": round(
            before_integrated_df["hybrid_risk_score"].mean(),
            3,
        ),
        "after_average_risk": round(
            after_integrated_df["hybrid_risk_score"].mean(),
            3,
        ),
        "remediated_identities": len(remediation_effect_df),
        "improved_identities": int(
            remediation_effect_df["risk_improved"].sum()
        ),
    }])


def validate_post_risk_outputs(
    before_integrated_df,
    after_integrated_df,
    comparison_df,
    remediation_effect_df,
):
    """조치 후 Risk 결과의 Identity·AI·위험도 계약을 검증합니다."""
    if after_integrated_df["identity"].duplicated().any():
        raise ValueError("after_integrated_df에 중복 identity가 있습니다.")
    if set(after_integrated_df["identity"]) != set(
        before_integrated_df["identity"]
    ):
        raise ValueError("조치 전후 Identity 집합이 다릅니다.")

    ai_score_unchanged = (
        comparison_df["before_ai_score"].round(12)
        == comparison_df["after_ai_score"].round(12)
    ).all()
    if not ai_score_unchanged:
        raise ValueError("행동 로그가 같은데 AI 점수가 변경되었습니다.")

    risk_not_increased = (
        remediation_effect_df["after_risk_score"]
        <= remediation_effect_df["before_risk_score"]
    ).all()
    if not risk_not_increased:
        raise ValueError("조치한 Identity의 위험도가 증가했습니다.")


def run_post_remediation_risk_comparison(
    before_integrated_df,
    before_ranked_df,
    ai_analysis_df,
    post_rule_summary_df,
    post_graph_summary_df,
    post_graph_findings_df,
    post_raw_findings_df,
    remediated_metadata_df,
    remediation_actions_df,
    merge_module_class,
    run_integrated_risk_analysis_func,
):
    """Cell 10-A 모듈을 재사용해 조치 후 Risk와 개선 효과를 계산합니다."""
    dataframe_inputs = {
        "integrated_risk_df": before_integrated_df,
        "risk_ranked_df": before_ranked_df,
        "ai_analysis_df": ai_analysis_df,
        "post_rule_identity_summary_df": post_rule_summary_df,
        "post_graph_identity_summary_df": post_graph_summary_df,
        "post_graph_findings_df": post_graph_findings_df,
        "post_raw_privilege_findings_df": post_raw_findings_df,
        "remediated_identity_metadata": remediated_metadata_df,
        "remediation_actions_df": remediation_actions_df,
    }
    validate_post_risk_inputs(
        dataframe_inputs,
        merge_module_class,
        run_integrated_risk_analysis_func,
    )

    integration_state = run_integrated_risk_analysis_func(
        rule_summary_df=post_rule_summary_df,
        ai_analysis_df=ai_analysis_df,
        graph_summary_df=post_graph_summary_df,
        graph_findings_df=post_graph_findings_df,
        raw_findings_df=post_raw_findings_df,
        metadata_df=remediated_metadata_df,
        merge_module_class=merge_module_class,
    )

    post_integrated_df = normalize_post_integrated_output(
        integration_state["integrated_risk_df"]
    )
    post_ranked_df = rank_post_remediation_risk(post_integrated_df)

    graph_agg_df, graph_finding_agg_df, raw_finding_agg_df = (
        prepare_post_graph_debug_outputs(
            post_graph_summary_df,
            post_graph_findings_df,
            post_raw_findings_df,
        )
    )

    comparison_state = build_risk_comparison(
        before_integrated_df,
        post_integrated_df,
        before_ranked_df,
        remediation_actions_df,
    )
    before_after_summary_df = create_before_after_summary(
        before_integrated_df,
        post_integrated_df,
        comparison_state["remediation_effect_df"],
    )

    validate_post_risk_outputs(
        before_integrated_df,
        post_integrated_df,
        comparison_state["comparison_df"],
        comparison_state["remediation_effect_df"],
    )

    return {
        "integration_state": integration_state,
        "rule_agg_df": integration_state["rule_agg_df"],
        "ai_agg_df": integration_state["ai_agg_df"],
        "graph_agg_df": graph_agg_df,
        "graph_finding_agg_df": graph_finding_agg_df,
        "raw_finding_agg_df": raw_finding_agg_df,
        "identity_base_df": integration_state["all_identities"],
        "merge_module": integration_state["merge_module"],
        "integrated_risk_df": post_integrated_df,
        "risk_ranked_df": post_ranked_df,
        **comparison_state,
        "before_after_summary_df": before_after_summary_df,
    }


print("Cell 12-4A completed: post-remediation Risk functions defined.")



In [ ]:

print("--- Cell 12-4B: Execute Post-remediation Risk Comparison ---")

post_risk_state = run_post_remediation_risk_comparison(
    before_integrated_df=integrated_risk_df,
    before_ranked_df=risk_ranked_df,
    ai_analysis_df=ai_analysis_df,
    post_rule_summary_df=post_rule_identity_summary_df,
    post_graph_summary_df=post_graph_identity_summary_df,
    post_graph_findings_df=post_graph_findings_df,
    post_raw_findings_df=post_raw_privilege_findings_df,
    remediated_metadata_df=remediated_identity_metadata,
    remediation_actions_df=remediation_actions_df,
    merge_module_class=MergeModule,
    run_integrated_risk_analysis_func=run_integrated_risk_analysis,
)

# 기존 Cell 12-4 변수명 유지
post_rule_agg_df = post_risk_state["rule_agg_df"]
post_ai_agg_df = post_risk_state["ai_agg_df"]
post_graph_agg_df = post_risk_state["graph_agg_df"]
graph_finding_agg_df = post_risk_state["graph_finding_agg_df"]
raw_finding_agg_df = post_risk_state["raw_finding_agg_df"]
post_identity_base_df = post_risk_state["identity_base_df"]
post_merge_module = post_risk_state["merge_module"]
post_integrated_risk_df = post_risk_state["integrated_risk_df"]
post_risk_ranked_df = post_risk_state["risk_ranked_df"]
before_df = post_risk_state["before_df"]
after_df = post_risk_state["after_df"]
applied_identities = post_risk_state["applied_identities"]
before_rank_map = post_risk_state["before_rank_map"]
risk_comparison_df = post_risk_state["comparison_df"]
remediation_effect_df = post_risk_state["remediation_effect_df"]
before_after_summary_df = post_risk_state["before_after_summary_df"]

# 기존 상수성 변수명도 유지
numeric_columns = POST_RISK_NUMERIC_COLUMNS.copy()
integer_columns = POST_RISK_INTEGER_COLUMNS.copy()
list_columns = POST_RISK_LIST_COLUMNS.copy()
action_order = POST_ACTION_ORDER.copy()
comparison_columns = RISK_COMPARISON_COLUMNS.copy()

print("\n1. Before-and-After Risk Comparison")
print("-----------------------------------")
display(remediation_effect_df[comparison_columns])

print("\n2. Overall Remediation Summary")
print("------------------------------")
display(before_after_summary_df)

print("All Cell 12-4 output contracts verified.")
print(
    "Cell 12-4 Post-remediation Risk "
    "Integration and Comparison completed successfully."
)
